<a href="https://colab.research.google.com/github/MTuan1510/Air-Project/blob/main/03_clean_chennai_hyderabad_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Missing values: Chennai & Hyderabad (Google Colab)
Làm theo quy trình của `01_clean_delhi_gurugram.ipynb`, dùng chung `src/utils.py` của repo.
Bước **Outliers** để làm sau.

**Quy ước của nhóm:** chuỗi 0 liên tiếp ≥ 3 ngày → NaN · chỉ nội suy khoảng thiếu ≤ 14 ngày · khối trống dài giữ NaN · không tự điền `AQI`.

## 0. Lấy code và dữ liệu từ GitHub

In [1]:
import os, sys

REPO = "/content/Air-Project"
if not os.path.exists(REPO):
    !git clone https://github.com/MTuan1510/Air-Project.git {REPO}
else:
    !git -C {REPO} pull

sys.path.append(f"{REPO}/src")
RAW = f"{REPO}/data/raw/city_day.csv"
INTERIM = f"{REPO}/data/interim"
os.makedirs(INTERIM, exist_ok=True)
print("Có file dữ liệu:", os.path.exists(RAW))

Cloning into '/content/Air-Project'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 24 (delta 2), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 875.21 KiB | 8.50 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Có file dữ liệu: True


In [2]:
import pandas as pd
import numpy as np
from utils import audit, load_and_split_city, diagnose_missing, impute_time_series

POLL = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene']

chennai = load_and_split_city("Chennai", raw_path=RAW)
hyderabad = load_and_split_city("Hyderabad", raw_path=RAW)

print("=== CHENNAI (audit gốc) ===")
display(audit(chennai).sort_values("pct_missing", ascending=False))
print("\n=== HYDERABAD (audit gốc) ===")
display(audit(hyderabad).sort_values("pct_missing", ascending=False))

=== CHENNAI (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
Xylene,float64,2009,100.0,0
PM10,float64,1707,85.0,297
NH3,float64,462,23.0,1456
Benzene,float64,285,14.2,370
Toluene,float64,253,12.6,606
AQI,float64,125,6.2,234
AQI_Bucket,object,125,6.2,6
PM2.5,float64,117,5.8,1669
O3,float64,49,2.4,1624
NO,float64,36,1.8,1018



=== HYDERABAD (audit gốc) ===


,dtype,n_missing,pct_missing,n_unique
NH3,float64,360,17.9,1147
PM10,float64,355,17.7,1556
AQI,float64,126,6.3,227
AQI_Bucket,object,126,6.3,6
Toluene,float64,127,6.3,1098
Xylene,float64,127,6.3,514
Benzene,float64,127,6.3,430
PM2.5,float64,115,5.7,1695
NO,float64,28,1.4,1039
NO2,float64,28,1.4,1671


## 1. Chẩn đoán missing

In [3]:
print("=== CHENNAI - diagnose_missing ===")
display(diagnose_missing(chennai))
print("\n=== HYDERABAD - diagnose_missing ===")
display(diagnose_missing(hyderabad))

=== CHENNAI - diagnose_missing ===


,pct_missing,block_ratio_%,action
col,,,
PM2.5,5.8,69,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
PM10,85.0,95,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
NO,1.8,47,nội suy trực tiếp (missing quá ít)
NO2,1.8,47,nội suy trực tiếp (missing quá ít)
NOx,1.8,47,nội suy trực tiếp (missing quá ít)
NH3,23.0,89,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
CO,1.2,44,nội suy trực tiếp (missing quá ít)
SO2,1.8,47,nội suy trực tiếp (missing quá ít)
O3,2.4,35,nội suy trực tiếp



=== HYDERABAD - diagnose_missing ===


,pct_missing,block_ratio_%,action
col,,,
PM2.5,5.7,75,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
PM10,17.7,68,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
NO,1.4,68,nội suy trực tiếp (missing quá ít)
NO2,1.4,68,nội suy trực tiếp (missing quá ít)
NOx,0.3,14,nội suy trực tiếp (missing quá ít)
NH3,17.9,69,"có khối lớn — chỉ nội suy phần rải rác, giữ Na..."
CO,0.2,20,nội suy trực tiếp (missing quá ít)
SO2,1.3,73,nội suy trực tiếp (missing quá ít)
O3,1.3,73,nội suy trực tiếp (missing quá ít)


**Quyết định drop (cả nhóm cần đồng ý):**
- Chennai `Xylene`: thiếu 100% → drop.
- Chennai `PM10`: thiếu 85%, chỉ có dữ liệu từ 2019 → chưa đạt ngưỡng drop 90%, giữ NaN như PM10 của Gurugram.
- Hyderabad: không drop cột nào.

## 2. Xử lý missing

In [4]:
chennai_clean, chennai_flat = impute_time_series(chennai, "Chennai", drop_cols=["Xylene"])
print()
hyderabad_clean, hyderabad_flat = impute_time_series(hyderabad, "Hyderabad", drop_cols=[])

--- Xử lý Missing Values cho Chennai ---
  - CO: phát hiện 40 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Benzene: phát hiện 137 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 143 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Đã xoá cột (đề xuất, cần nhóm đồng ý): ['Xylene']
  => Còn 2453 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)

--- Xử lý Missing Values cho Hyderabad ---
  - NOx: phát hiện 18 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - CO: phát hiện 67 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Benzene: phát hiện 23 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Toluene: phát hiện 14 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  - Xylene: phát hiện 14 ngày flatline (0 liên tục >= 3 ngày) -> NaN
  => Còn 1191 ô missing SAU xử lý (CHỦ Ý giữ NaN ở khối trống dài, không bịa số)


## 3. Kiểm chứng trước / sau

In [5]:
def compare_missing(before_df, after_df, cols):
    before = before_df[cols].isna().mean() * 100
    after_cols = [c for c in cols if c in after_df.columns]
    after = after_df[after_cols].isna().mean() * 100
    return pd.DataFrame({'truoc_%': before.round(1), 'sau_%': after.reindex(before.index).round(1)})

print("=== CHENNAI ===");   display(compare_missing(chennai, chennai_clean, POLL + ['AQI']))
print("\n=== HYDERABAD ==="); display(compare_missing(hyderabad, hyderabad_clean, POLL + ['AQI']))

=== CHENNAI ===


,truoc_%,sau_%
PM2.5,5.8,4.2
PM10,85.0,83.1
NO,1.8,0.1
NO2,1.8,0.1
NOx,1.8,0.1
NH3,23.0,19.8
CO,1.2,0.0
SO2,1.8,0.1
O3,2.4,0.1
Benzene,14.2,5.3



=== HYDERABAD ===


,truoc_%,sau_%
PM2.5,5.7,4.5
PM10,17.7,15.2
NO,1.4,0.2
NO2,1.4,0.2
NOx,0.3,0.2
NH3,17.9,15.5
CO,0.2,1.6
SO2,1.3,0.2
O3,1.3,0.2
Benzene,6.3,5.0


## 4. Lưu và tải về máy

In [6]:
chennai_clean.to_csv(f"{INTERIM}/chennai_after_missing.csv", index=False)
hyderabad_clean.to_csv(f"{INTERIM}/hyderabad_after_missing.csv", index=False)
print("Chennai:", chennai_clean.shape, "| Hyderabad:", hyderabad_clean.shape)

from google.colab import files
files.download(f"{INTERIM}/chennai_after_missing.csv")
files.download(f"{INTERIM}/hyderabad_after_missing.csv")

Chennai: (2009, 28) | Hyderabad: (2006, 29)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>